# log-samples-eval-callback — worked example 3: Stop logging once the sink is full

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `log-samples-eval-callback`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A capacity-bounded callback adds a guard `len(sink) < max_entries` so it stops appending once the sink reaches its limit, even if the cadence would otherwise fire. This caps memory or upload volume for long runs.

## Worked solution

We combine cadence with a capacity ceiling.

1. **Cadence test.** Same `step % eval_every == 0` gate.
2. **Capacity guard.** Before appending we check `len(sink) < max_entries`. Once the sink is at capacity, further cadence hits are skipped. We can `break` because no later step will ever find room.
3. **Count only this call's fires.** We return the number of appends performed here, independent of any records the sink already held.
4. **Edge cases.** `max_entries == 0` means no fires; a pre-filled sink at capacity also yields none.

The demo runs many steps with `max_entries=3` and prints that exactly three records were logged despite the cadence allowing more.

In [ ]:
def run_with_cap(n_steps, eval_every, n_eval, sink, max_entries):
    n_fires = 0
    for step in range(n_steps):
        if step % eval_every != 0:
            continue
        if len(sink) >= max_entries:
            break
        samples = [f'step={step}-sample={i}' for i in range(n_eval)]
        sink.append({'step': step, 'samples': samples})
        n_fires += 1
    return n_fires

sink = []
fires = run_with_cap(n_steps=30, eval_every=2, n_eval=1, sink=sink, max_entries=3)
print('fires:', fires)
print('records:', len(sink))
print('fired at steps:', [d['step'] for d in sink])